# Diagnostics (Universal)

This notebook provides a universal diagnostics workflow that works for **any ML type**
(binary classification, multiclass, regression, time-to-event).

**Part 1 — Study Overview** (via `predict/notebook_utils`): study details, performance metrics, selected features.

**Part 2 — Feature Importances** (via `diagnostics`): saved FI parquet files, filterable by method, dataset, etc.

**Part 3 — Optuna Diagnostics** (via `diagnostics`): trial counts, trials scatter, hyperparameters.

No model loading is performed.

## Imports

In [ ]:
from octopus.predict.notebook_utils import (
    find_latest_study,
    show_study_details,
    show_target_metric_performance,
    show_selected_features,
)
from octopus.diagnostics import StudyDiagnostics

## Input

In [ ]:
study_name_prefix = "wf_octo_mrmr_octo"
studies_root = "../studies"

study_directory = find_latest_study(studies_root, study_name_prefix)
print(f"Using study: {study_directory}")

---
## Part 1: Study Overview

### Study Details

In [ ]:
study_info = show_study_details(study_directory)

### Target Metric Performance

In [ ]:
performance_tables = show_target_metric_performance(study_info)

### Selected Features Summary

In [ ]:
feature_table, freq_table, raw = show_selected_features(study_info)

---
## Part 2: Feature Importances (from saved results)

Two-step workflow:
1. **`get_feature_importances()`** — load raw FI data filtered by task/module (returns a DataFrame)
2. **`plot_feature_importance()`** — plot the FI data with additional filters (fi_method, fi_dataset, etc.)

In [ ]:
diag = StudyDiagnostics(study_info["path"])

### Load raw FI data for task 0

In [ ]:
fi_table = diag.get_feature_importances(task_id=0, result_type="best")
print(f"FI rows: {len(fi_table)}")
for col in ["fi_method", "fi_dataset", "training_id", "module", "outersplit_id"]:
    if col in fi_table.columns:
        print(f"  {col}: {sorted(fi_table[col].unique().tolist())}")
fi_table.head()

### Internal Feature Importance (train set)

In [ ]:
diag.plot_feature_importance(fi_table, fi_method="internal", fi_dataset="train")

### Permutation Feature Importance (dev set)

In [ ]:
diag.plot_feature_importance(fi_table, fi_method="permutation", fi_dataset="dev")

### Single outersplit view

In [ ]:
diag.plot_feature_importance(fi_table, fi_method="internal", outersplit_id=0, top_n=15)

---
## Part 3: Optuna Diagnostics

### Number of Unique Trials by Model Type

In [ ]:
diag.plot_optuna_trial_counts()

### Optuna Trials: Objective Value and Best Value

In [ ]:
diag.plot_optuna_trials(outersplit_id=0, task_id=0)

### Optuna Hyperparameters

In [ ]:
diag.plot_optuna_hyperparameters(outersplit_id=0, task_id=0)